# 03 Poisoned retraining and evaluation

This notebook covers three pieces of the workflow:
1. Add the generated poisons to the training set with the Bird label.
2. Retrain the model from scratch.
3. Check whether held-out Plane targets are classified as Bird.

Inputs: `step1_data.pt` and `poisons.pt` from the previous notebooks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, ConcatDataset, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load data and poisons

In [ ]:
# Load metadata
data_payload = torch.load('step1_data.pt')
poisons = torch.load('poisons.pt')

clean_train_indices = data_payload['clean_train_indices']
test_indices = data_payload['test_indices']
target_indices_global = data_payload['target_indices_global']
base_indices_global = data_payload['base_indices_global']
TARGET_CLASS_IDX = data_payload['target_class_idx']
BASE_CLASS_IDX = data_payload['base_class_idx']
CLASS_NAMES = data_payload['class_names']

# Load datasets
transform = transforms.Compose([transforms.ToTensor()])
trainset_full = torchvision.datasets.CIFAR10(root='./data/cifar-10-python', train=True, download=False, transform=transform)
testset_full = torchvision.datasets.CIFAR10(root='./data/cifar-10-python', train=False, download=False, transform=transform)

clean_train_subset = Subset(trainset_full, clean_train_indices)
test_subset = Subset(testset_full, test_indices)

# Create poison dataset
# The poisons keep the base class label (Bird = 2)
poison_labels = torch.full((len(poisons),), BASE_CLASS_IDX, dtype=torch.long)
poison_dataset = TensorDataset(poisons.cpu(), poison_labels)

# Combine clean data and poisons
poisoned_trainset = ConcatDataset([clean_train_subset, poison_dataset])

poisoned_loader = DataLoader(poisoned_trainset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

print(f"Clean Train Size: {len(clean_train_subset)}")
print(f"Poisons Added: {len(poison_dataset)}")
print(f"Total Poisoned Train Size: {len(poisoned_trainset)}")

## Define the same model architecture

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

## Retrain on poisoned data

In [ ]:
def train_model(model, loader, epochs=15):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(loader):.4f}")

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# Initialize a new model
poisoned_model = SimpleCNN().to(device)
print("Retraining on Poisoned Data...")
# Train for 15 epochs to match the reported run
train_model(poisoned_model, poisoned_loader, epochs=15)

# Check overall accuracy
acc = evaluate(poisoned_model, test_loader)
print(f"Poisoned Model Test Accuracy: {acc*100:.2f}%")

## Check attack success
Evaluate the original held-out target images after retraining.

Success means a held-out Plane target is now classified as Bird.

In [ ]:
# Load targets
targets_list = [trainset_full[i][0] for i in target_indices_global]
targets_tensor = torch.stack(targets_list).to(device)

poisoned_model.eval()
with torch.no_grad():
    out = poisoned_model(targets_tensor)
    _, preds = torch.max(out, 1)

print("--- Attack Results ---")
print(f"Predictions on Targets: {preds.cpu().numpy()}")
print(f"Original True Class: {TARGET_CLASS_IDX} ({CLASS_NAMES[TARGET_CLASS_IDX]})")
print(f"Target Attack Class: {BASE_CLASS_IDX} ({CLASS_NAMES[BASE_CLASS_IDX]})")

num_success = (preds == BASE_CLASS_IDX).sum().item()
print(f"\nAttack Success Rate: {num_success}/{len(preds)} ({num_success/len(preds)*100:.1f}%)")

if num_success > 0:
    print("SUCCESS: At least one target was misclassified as the Base class!")
else:
    print("FAILURE: Targets were not misclassified. (This is common with basic Feature Collision on deep networks or insufficient epochs/poisons).")

## Class-wise statistics
Class-wise accuracy helps show whether the attack damaged the model beyond the selected targets.

In [ ]:
class_correct = list(0. for i in range(4))
class_total = list(0. for i in range(4))

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = poisoned_model(inputs)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(len(labels)):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

print("Class-wise Accuracy:")
for i in range(4):
    print(f"{CLASS_NAMES[i]}: {100 * class_correct[i] / class_total[i]:.2f}%")